# 05. Content Feature Engineering

이 노트북의 목적은 `03_movie_metadata_unification.ipynb`에서 생성한 `movie_metadata_unified_v2.csv`를 `02_preprocessing_policy.ipynb`의 3주 관측창 시청이력에 붙여, 구독 이벤트 단위의 콘텐츠 성향 파생변수를 생성하는 것이다.

이 버전은 `04_usage_feature_engineering.ipynb`의 watch gap feature 패치가 반영된 산출물, 즉 `modeling_feature_table_usage.csv`가 정상 생성되어 있다는 전제에서 실행한다. 04번의 원본 오류는 05번 설계 자체의 문제는 아니지만, 05번은 04번의 최종 산출물을 입력으로 받기 때문에 패치본 04번 이후 실행하는 것을 기준으로 한다.

핵심 원칙은 다음과 같다.

1. 분석 단위는 `membership_row_id`이다.
2. 시청이력은 02번에서 만든 고객별 `reg_date` 기준 day 0~20 관측창만 사용한다.
3. 영화 메타데이터는 03번의 `movie_metadata_unified_v2.csv`를 사용한다.
4. KOBIS 저신뢰 매칭은 CSV에는 남겨두되, `use_for_content_features == 0`이면 콘텐츠 피처 계산에서 제외한다.
5. 장르는 대표 장르 순서가 없으므로 `top1` 방식으로 강제 선택하지 않는다.
6. 장르는 콘텐츠 성향 태그로 보고, 멀티핫 방식의 `tag_ratio_*`를 중심으로 생성한다.
7. 균등 배분 방식의 `alloc_ratio_*`는 보조 검증 변수로 생성한다.
8. 콘텐츠 피처는 이탈의 단독 원인으로 단정하지 않고, 100원딜/요금제/시청패턴 세그먼트를 설명하는 보조 변수로 사용한다.


## 5-1. 라이브러리 로딩


In [ ]:
from pathlib import Path
import json
import math
import re
from collections import Counter

import numpy as np
import pandas as pd


## 5-2. 경로 설정

이 노트북은 `park.ingyeom` 폴더 안에서 실행하는 것을 기준으로 한다. 02번, 03번, 04번 노트북이 먼저 실행되어 있어야 한다.


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd()
    start = start.resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / '_data').exists() and (p / 'notebooks').exists():
            return p
        if p.name == 'park.ingyeom':
            return p
    return start

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / '_data'
RAW_DIR = DATA_DIR / '01_raw'
INTERIM_DIR = DATA_DIR / '02_interim'
PROCESSED_DIR = DATA_DIR / '03_processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
TABLES_DIR = REPORTS_DIR / 'tables'

for d in [INTERIM_DIR, PROCESSED_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('INTERIM_DIR:', INTERIM_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('TABLES_DIR:', TABLES_DIR)


## 5-3. 입력 파일 로딩

05번은 원본 파일을 다시 전처리하지 않는다. 반드시 앞 단계 산출물을 입력으로 사용한다.

필수 입력은 다음과 같다.

- `_data/02_interim/view_history_observation_window.csv`
- `_data/02_interim/movie_metadata_unified_v2.csv`
- `_data/03_processed/modeling_feature_table_usage.csv`

`modeling_feature_table_usage.csv`는 04번 패치본 기준으로 생성되어야 한다. 특히 watch gap feature가 `groupby().apply(...).unstack().reset_index()` 방식으로 wide table로 펴진 결과여야 한다.


In [ ]:
INPUT_FILES = {
    'obs_view': INTERIM_DIR / 'view_history_observation_window.csv',
    'movie_metadata_v2': INTERIM_DIR / 'movie_metadata_unified_v2.csv',
    'modeling_usage': PROCESSED_DIR / 'modeling_feature_table_usage.csv',
}

missing = [str(p) for p in INPUT_FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        '05번 실행 전 02, 03, 04번 노트북 산출물이 필요합니다. Missing: ' + str(missing)
    )

obs_view = pd.read_csv(INPUT_FILES['obs_view'])
movie_meta = pd.read_csv(INPUT_FILES['movie_metadata_v2'])
modeling_usage = pd.read_csv(INPUT_FILES['modeling_usage'])

if 'watch_time(min)' in obs_view.columns and 'watch_time' not in obs_view.columns:
    obs_view = obs_view.rename(columns={'watch_time(min)': 'watch_time'})

if 'watch_day' in obs_view.columns:
    obs_view['watch_day'] = pd.to_datetime(obs_view['watch_day'])

file_summary = pd.DataFrame([
    {'name': 'view_history_observation_window', 'path': str(INPUT_FILES['obs_view']), 'rows': len(obs_view), 'cols': obs_view.shape[1]},
    {'name': 'movie_metadata_unified_v2', 'path': str(INPUT_FILES['movie_metadata_v2']), 'rows': len(movie_meta), 'cols': movie_meta.shape[1]},
    {'name': 'modeling_feature_table_usage', 'path': str(INPUT_FILES['modeling_usage']), 'rows': len(modeling_usage), 'cols': modeling_usage.shape[1]},
])
file_summary.to_csv(TABLES_DIR / '05_content_feature_input_file_summary.csv', index=False, encoding='utf-8-sig')
display(file_summary)


## 5-4. 필수 컬럼 검산


In [ ]:
required_obs_cols = {'membership_row_id', 'MOVIE_NUM', 'watch_time'}
required_meta_cols = {
    'MOVIE_NUM', 'movie_title', 'metadata_source', 'metadata_quality',
    'use_for_content_features', 'metadata_covered_flag',
    'unified_genres', 'unified_countries', 'unified_age_rating',
    'unified_runtime_min', 'unified_release_year',
}
required_usage_cols = {'membership_row_id', 'is_repurchase', 'is_100won', 'max_screen'}

missing_obs_cols = sorted(required_obs_cols - set(obs_view.columns))
missing_meta_cols = sorted(required_meta_cols - set(movie_meta.columns))
missing_usage_cols = sorted(required_usage_cols - set(modeling_usage.columns))

if missing_obs_cols:
    raise ValueError(f'obs_view 필수 컬럼 누락: {missing_obs_cols}')
if missing_meta_cols:
    raise ValueError(f'movie_metadata_v2 필수 컬럼 누락: {missing_meta_cols}')
if missing_usage_cols:
    raise ValueError(f'modeling_usage 필수 컬럼 누락: {missing_usage_cols}')

input_check = pd.DataFrame([
    {'check': 'obs_view_membership_row_id_unique', 'value': int(obs_view['membership_row_id'].nunique())},
    {'check': 'obs_view_rows', 'value': int(len(obs_view))},
    {'check': 'obs_view_unique_movies', 'value': int(obs_view['MOVIE_NUM'].nunique())},
    {'check': 'movie_meta_rows', 'value': int(len(movie_meta))},
    {'check': 'movie_meta_unique_movies', 'value': int(movie_meta['MOVIE_NUM'].nunique())},
    {'check': 'modeling_usage_rows', 'value': int(len(modeling_usage))},
    {'check': 'modeling_usage_unique_membership_row_id', 'value': int(modeling_usage['membership_row_id'].nunique())},
])
input_check.to_csv(TABLES_DIR / '05_content_feature_input_key_check.csv', index=False, encoding='utf-8-sig')
display(input_check)


## 5-5. 시청이력과 영화 메타데이터 연결

`View_History.MOVIE_NUM`과 `movie_metadata_unified_v2.MOVIE_NUM`을 기준으로 연결한다.

메타데이터가 없거나 저신뢰로 판단된 영화는 join 결과에는 남긴다. 다만 콘텐츠 성향 피처 계산에서는 `use_for_content_features == 1`인 행만 사용한다.


In [ ]:
obs = obs_view.copy()
meta = movie_meta.copy()

obs['MOVIE_NUM'] = pd.to_numeric(obs['MOVIE_NUM'], errors='coerce').astype('Int64')
meta['MOVIE_NUM'] = pd.to_numeric(meta['MOVIE_NUM'], errors='coerce').astype('Int64')
obs['watch_time'] = pd.to_numeric(obs['watch_time'], errors='coerce').fillna(0)

view_meta = obs.merge(meta, on='MOVIE_NUM', how='left', suffixes=('', '_movie'))

# join되지 않은 영화는 missing 처리한다.
view_meta['metadata_covered_flag'] = view_meta['metadata_covered_flag'].fillna(0).astype(int)
view_meta['use_for_content_features'] = view_meta['use_for_content_features'].fillna(0).astype(int)
view_meta['metadata_source'] = view_meta['metadata_source'].fillna('missing_after_join')
view_meta['metadata_quality'] = view_meta['metadata_quality'].fillna('E_missing_after_join')

join_summary = pd.DataFrame([
    {'metric': 'obs_view_rows', 'value': int(len(obs))},
    {'metric': 'view_meta_rows', 'value': int(len(view_meta))},
    {'metric': 'unique_movies_in_obs', 'value': int(obs['MOVIE_NUM'].nunique())},
    {'metric': 'unique_movies_joined_usable', 'value': int(view_meta.loc[view_meta['use_for_content_features'] == 1, 'MOVIE_NUM'].nunique())},
    {'metric': 'watch_time_total', 'value': float(view_meta['watch_time'].sum())},
    {'metric': 'watch_time_metadata_covered', 'value': float(view_meta.loc[view_meta['metadata_covered_flag'] == 1, 'watch_time'].sum())},
    {'metric': 'watch_time_usable_for_content_features', 'value': float(view_meta.loc[view_meta['use_for_content_features'] == 1, 'watch_time'].sum())},
])
join_summary.to_csv(TABLES_DIR / '05_content_feature_view_metadata_join_summary.csv', index=False, encoding='utf-8-sig')
display(join_summary)


## 5-6. 메타데이터 커버리지 피처

콘텐츠 피처는 영화 메타데이터가 붙은 시청기록에만 의존한다. 따라서 유저별로 메타데이터 커버율 자체를 피처로 남긴다.


In [ ]:
def safe_divide(numer, denom):
    return np.where(denom > 0, numer / denom, 0)

coverage_features = view_meta.groupby('membership_row_id').agg(
    total_watch_time_for_content=('watch_time', 'sum'),
    content_total_sessions=('watch_time', 'count'),
    content_unique_movies=('MOVIE_NUM', 'nunique'),
    metadata_covered_watch_time=('watch_time', lambda s: s[view_meta.loc[s.index, 'metadata_covered_flag'] == 1].sum()),
    usable_metadata_watch_time=('watch_time', lambda s: s[view_meta.loc[s.index, 'use_for_content_features'] == 1].sum()),
).reset_index()

coverage_features['metadata_covered_watch_ratio'] = safe_divide(
    coverage_features['metadata_covered_watch_time'],
    coverage_features['total_watch_time_for_content'],
)
coverage_features['usable_metadata_watch_ratio'] = safe_divide(
    coverage_features['usable_metadata_watch_time'],
    coverage_features['total_watch_time_for_content'],
)
coverage_features['metadata_missing_watch_time'] = (
    coverage_features['total_watch_time_for_content'] - coverage_features['metadata_covered_watch_time']
)
coverage_features['metadata_missing_watch_ratio'] = safe_divide(
    coverage_features['metadata_missing_watch_time'],
    coverage_features['total_watch_time_for_content'],
)

coverage_features.head()


## 5-7. 문자열 분리와 컬럼명 정리 함수


In [ ]:
def split_multi(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return []
    parts = re.split(r'[|,]+', text)
    return [p.strip() for p in parts if p.strip()]


def safe_col_name(value: str) -> str:
    text = str(value).strip()
    text = text.replace('/', '_')
    text = text.replace(' ', '_')
    text = text.replace('-', '_')
    text = re.sub(r'[^0-9A-Za-z가-힣_]+', '', text)
    text = re.sub(r'_+', '_', text).strip('_')
    return text


def entropy_from_values(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[arr > 0]
    if arr.size == 0:
        return 0.0
    p = arr / arr.sum()
    return float(-(p * np.log(p)).sum())


def normalized_entropy_from_values(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[arr > 0]
    if arr.size <= 1:
        return 0.0
    ent = entropy_from_values(arr)
    return float(ent / np.log(arr.size))


## 5-8. 장르 피처 생성

장르 피처는 두 방식으로 만든다.

1. `tag_dur_*`, `tag_ratio_*`: multi-hot 태그 방식이다. 어떤 영화를 100분 봤고 장르가 `액션|스릴러/범죄`이면 액션 100분, 스릴러/범죄 100분으로 계산한다. 합이 실제 시청시간을 넘을 수 있지만, 콘텐츠 성향 태그로는 이 방식이 더 자연스럽다.
2. `alloc_dur_*`, `alloc_ratio_*`: 균등 배분 방식이다. 같은 예시에서 액션 50분, 스릴러/범죄 50분으로 계산한다. 합이 실제 시청시간과 맞기 때문에 다양성/entropy 계산에 적합하다.


In [ ]:
usable = view_meta.loc[view_meta['use_for_content_features'] == 1].copy()
usable['genre_list'] = usable['unified_genres'].map(split_multi)

# multi-hot genre durations
rows = []
for row in usable[['membership_row_id', 'watch_time', 'genre_list']].itertuples(index=False):
    if not row.genre_list:
        continue
    for genre in row.genre_list:
        rows.append((row.membership_row_id, genre, row.watch_time, row.watch_time / len(row.genre_list)))

genre_long = pd.DataFrame(rows, columns=['membership_row_id', 'genre', 'tag_watch_time', 'alloc_watch_time'])

if len(genre_long) > 0:
    tag_genre = genre_long.pivot_table(
        index='membership_row_id', columns='genre', values='tag_watch_time', aggfunc='sum', fill_value=0
    )
    tag_genre.columns = [f'tag_dur_{safe_col_name(c)}' for c in tag_genre.columns]
    tag_genre = tag_genre.reset_index()

    alloc_genre = genre_long.pivot_table(
        index='membership_row_id', columns='genre', values='alloc_watch_time', aggfunc='sum', fill_value=0
    )
    alloc_genre.columns = [f'alloc_dur_{safe_col_name(c)}' for c in alloc_genre.columns]
    alloc_genre = alloc_genre.reset_index()
else:
    tag_genre = pd.DataFrame(columns=['membership_row_id'])
    alloc_genre = pd.DataFrame(columns=['membership_row_id'])

# genre count and entropy from allocation durations
genre_stats = genre_long.groupby('membership_row_id').agg(
    genre_tag_count=('genre', 'nunique')
).reset_index() if len(genre_long) > 0 else pd.DataFrame(columns=['membership_row_id', 'genre_tag_count'])

if len(genre_long) > 0:
    entropy_rows = []
    for mid, g in genre_long.groupby('membership_row_id'):
        alloc_by_genre = g.groupby('genre')['alloc_watch_time'].sum().values
        entropy_rows.append({
            'membership_row_id': mid,
            'genre_tag_entropy': entropy_from_values(alloc_by_genre),
            'genre_tag_entropy_norm': normalized_entropy_from_values(alloc_by_genre),
        })
    genre_entropy = pd.DataFrame(entropy_rows)
else:
    genre_entropy = pd.DataFrame(columns=['membership_row_id', 'genre_tag_entropy', 'genre_tag_entropy_norm'])

# top genre from tag duration
if len(genre_long) > 0:
    top_genre = (
        genre_long.groupby(['membership_row_id', 'genre'])['tag_watch_time']
        .sum()
        .reset_index()
        .sort_values(['membership_row_id', 'tag_watch_time', 'genre'], ascending=[True, False, True])
        .drop_duplicates('membership_row_id')
        .rename(columns={'genre': 'top_genre_tag', 'tag_watch_time': 'top_genre_tag_watch_time'})
    )
else:
    top_genre = pd.DataFrame(columns=['membership_row_id', 'top_genre_tag', 'top_genre_tag_watch_time'])

print('genre_long rows:', len(genre_long))
print('tag_genre shape:', tag_genre.shape)
print('alloc_genre shape:', alloc_genre.shape)


## 5-9. 관람등급 피처 생성


In [ ]:
usable['rating_list'] = usable['unified_age_rating'].map(split_multi)

rating_rows = []
for row in usable[['membership_row_id', 'watch_time', 'rating_list']].itertuples(index=False):
    if not row.rating_list:
        continue
    for rating in row.rating_list:
        rating_rows.append((row.membership_row_id, rating, row.watch_time, row.watch_time / len(row.rating_list)))

rating_long = pd.DataFrame(rating_rows, columns=['membership_row_id', 'rating', 'tag_watch_time', 'alloc_watch_time'])

if len(rating_long) > 0:
    rating_features = rating_long.pivot_table(
        index='membership_row_id', columns='rating', values='alloc_watch_time', aggfunc='sum', fill_value=0
    )
    rating_features.columns = [f'rating_dur_{safe_col_name(c)}' for c in rating_features.columns]
    rating_features = rating_features.reset_index()
else:
    rating_features = pd.DataFrame(columns=['membership_row_id'])

rating_stats = rating_long.groupby('membership_row_id').agg(
    rating_tag_count=('rating', 'nunique')
).reset_index() if len(rating_long) > 0 else pd.DataFrame(columns=['membership_row_id', 'rating_tag_count'])

print('rating_long rows:', len(rating_long))
print('rating_features shape:', rating_features.shape)


## 5-10. 국가 피처 생성


In [ ]:
usable['country_list'] = usable['unified_countries'].map(split_multi)

country_rows = []
for row in usable[['membership_row_id', 'watch_time', 'country_list']].itertuples(index=False):
    if not row.country_list:
        continue
    for country in row.country_list:
        country_rows.append((row.membership_row_id, country, row.watch_time, row.watch_time / len(row.country_list)))

country_long = pd.DataFrame(country_rows, columns=['membership_row_id', 'country', 'tag_watch_time', 'alloc_watch_time'])

# 국가 전체를 전부 column으로 만들면 과도해질 수 있으므로 주요 국가만 explicit feature로 만든다.
COUNTRY_MAP = {
    '한국': 'korean',
    '미국': 'us',
    '일본': 'japanese',
    '중국': 'chinese',
    '홍콩': 'hongkong',
    '영국': 'uk',
    '프랑스': 'french',
}

if len(country_long) > 0:
    country_long['country_group'] = country_long['country'].map(COUNTRY_MAP).fillna('other_foreign')
    country_features = country_long.pivot_table(
        index='membership_row_id', columns='country_group', values='alloc_watch_time', aggfunc='sum', fill_value=0
    )
    country_features.columns = [f'country_dur_{safe_col_name(c)}' for c in country_features.columns]
    country_features = country_features.reset_index()
else:
    country_features = pd.DataFrame(columns=['membership_row_id'])

country_stats = country_long.groupby('membership_row_id').agg(
    country_tag_count=('country', 'nunique')
).reset_index() if len(country_long) > 0 else pd.DataFrame(columns=['membership_row_id', 'country_tag_count'])

if len(country_long) > 0:
    country_entropy_rows = []
    for mid, g in country_long.groupby('membership_row_id'):
        alloc_by_country = g.groupby('country')['alloc_watch_time'].sum().values
        country_entropy_rows.append({
            'membership_row_id': mid,
            'country_entropy': entropy_from_values(alloc_by_country),
            'country_entropy_norm': normalized_entropy_from_values(alloc_by_country),
        })
    country_entropy = pd.DataFrame(country_entropy_rows)
else:
    country_entropy = pd.DataFrame(columns=['membership_row_id', 'country_entropy', 'country_entropy_norm'])

print('country_long rows:', len(country_long))
print('country_features shape:', country_features.shape)


## 5-11. 플래그형 콘텐츠 피처와 가중 평균 피처

영화 단위 통합 메타데이터에 있는 boolean flag와 수치형 메타데이터를 시청시간 가중 방식으로 유저 단위에 집계한다.


In [ ]:
flag_cols = [
    'is_kids_animation', 'is_family_content', 'is_adult_content',
    'is_korean_content', 'is_us_content', 'is_japanese_content',
    'is_recent_content', 'is_old_content', 'is_long_movie', 'is_short_content',
]
flag_cols = [c for c in flag_cols if c in usable.columns]

for c in flag_cols:
    usable[c] = pd.to_numeric(usable[c], errors='coerce').fillna(0).astype(int)

flag_feature_parts = []
for c in flag_cols:
    tmp = usable.assign(weighted_flag=usable['watch_time'] * usable[c])
    agg = tmp.groupby('membership_row_id')['weighted_flag'].sum().reset_index()
    agg = agg.rename(columns={'weighted_flag': f'{c}_watch_time'})
    flag_feature_parts.append(agg)

flag_features = coverage_features[['membership_row_id']].copy()
for part in flag_feature_parts:
    flag_features = flag_features.merge(part, on='membership_row_id', how='left')

for c in [col for col in flag_features.columns if col.endswith('_watch_time')]:
    flag_features[c] = flag_features[c].fillna(0)
    ratio_col = c.replace('_watch_time', '_ratio')
    flag_features[ratio_col] = safe_divide(flag_features[c], coverage_features.set_index('membership_row_id').loc[flag_features['membership_row_id'], 'total_watch_time_for_content'].values)

# Weighted runtime and release year.
num = usable.copy()
num['unified_runtime_min'] = pd.to_numeric(num['unified_runtime_min'], errors='coerce')
num['unified_release_year'] = pd.to_numeric(num['unified_release_year'], errors='coerce')

weighted_rows = []
for mid, g in num.groupby('membership_row_id'):
    row = {'membership_row_id': mid}
    g_runtime = g.dropna(subset=['unified_runtime_min'])
    if len(g_runtime) and g_runtime['watch_time'].sum() > 0:
        row['avg_runtime_weighted'] = float((g_runtime['unified_runtime_min'] * g_runtime['watch_time']).sum() / g_runtime['watch_time'].sum())
    else:
        row['avg_runtime_weighted'] = 0.0

    g_year = g.dropna(subset=['unified_release_year'])
    if len(g_year) and g_year['watch_time'].sum() > 0:
        row['avg_release_year_weighted'] = float((g_year['unified_release_year'] * g_year['watch_time']).sum() / g_year['watch_time'].sum())
        row['avg_content_age_from_2021_weighted'] = float(((2021 - g_year['unified_release_year']) * g_year['watch_time']).sum() / g_year['watch_time'].sum())
    else:
        row['avg_release_year_weighted'] = 0.0
        row['avg_content_age_from_2021_weighted'] = 0.0
    weighted_rows.append(row)

weighted_numeric_features = pd.DataFrame(weighted_rows)
print('flag_features shape:', flag_features.shape)
print('weighted_numeric_features shape:', weighted_numeric_features.shape)


## 5-12. 콘텐츠 피처 병합과 ratio 계산


In [ ]:
content_features = coverage_features.copy()

for part in [
    tag_genre, alloc_genre, genre_stats, genre_entropy, top_genre,
    rating_features, rating_stats,
    country_features, country_stats, country_entropy,
    flag_features, weighted_numeric_features,
]:
    if len(part.columns) > 1:
        content_features = content_features.merge(part, on='membership_row_id', how='left')

# 결측 채우기
text_cols = ['top_genre_tag']
for c in text_cols:
    if c in content_features.columns:
        content_features[c] = content_features[c].fillna('no_usable_metadata')

for c in content_features.columns:
    if c not in ['membership_row_id', 'top_genre_tag']:
        content_features[c] = pd.to_numeric(content_features[c], errors='coerce').fillna(0)

# tag/alloc/rating/country duration columns를 ratio로 변환한다.
denominator = content_features['total_watch_time_for_content'].replace(0, np.nan)
for prefix in ['tag_dur_', 'alloc_dur_', 'rating_dur_', 'country_dur_']:
    dur_cols = [c for c in content_features.columns if c.startswith(prefix)]
    for c in dur_cols:
        ratio_prefix = prefix.replace('_dur_', '_ratio_')
        ratio_col = c.replace(prefix, ratio_prefix)
        content_features[ratio_col] = (content_features[c] / denominator).fillna(0)

# 주요 국가 ratio alias
alias_map = {
    'country_ratio_korean': 'korean_content_ratio_from_country',
    'country_ratio_us': 'us_content_ratio_from_country',
    'country_ratio_japanese': 'japanese_content_ratio_from_country',
    'country_ratio_other_foreign': 'other_foreign_content_ratio_from_country',
}
for src, dst in alias_map.items():
    if src in content_features.columns:
        content_features[dst] = content_features[src]

print('content_features shape:', content_features.shape)
content_features.head()


## 5-13. 콘텐츠 피처 품질 요약


In [ ]:
content_numeric_cols = [c for c in content_features.columns if c not in ['membership_row_id', 'top_genre_tag']]
content_feature_summary = content_features[content_numeric_cols].describe().T.reset_index().rename(columns={'index': 'feature'})
content_feature_summary['missing_count'] = content_features[content_numeric_cols].isna().sum().values
content_feature_summary['zero_count'] = (content_features[content_numeric_cols] == 0).sum().values
content_feature_summary['zero_rate'] = content_feature_summary['zero_count'] / len(content_features)
content_feature_summary.to_csv(TABLES_DIR / '05_content_feature_numeric_summary.csv', index=False, encoding='utf-8-sig')

top_genre_counts = content_features['top_genre_tag'].value_counts(dropna=False).reset_index()
top_genre_counts.columns = ['top_genre_tag', 'count']
top_genre_counts.to_csv(TABLES_DIR / '05_content_feature_top_genre_counts.csv', index=False, encoding='utf-8-sig')

display(content_feature_summary.head(30))
display(top_genre_counts.head(20))


## 5-14. 04번 사용 행동 테이블과 콘텐츠 피처 결합

최종 모델링 후보 테이블은 04번의 `modeling_feature_table_usage.csv`에 05번 콘텐츠 피처를 붙여 만든다.


In [ ]:
modeling_with_content = modeling_usage.merge(content_features, on='membership_row_id', how='left')

# 시청이력이나 usable metadata가 없는 고객은 콘텐츠 수치형 피처를 0으로 채운다.
for c in content_features.columns:
    if c == 'membership_row_id':
        continue
    if c == 'top_genre_tag':
        modeling_with_content[c] = modeling_with_content[c].fillna('no_watch_or_no_usable_metadata')
    else:
        modeling_with_content[c] = pd.to_numeric(modeling_with_content[c], errors='coerce').fillna(0)

modeling_with_content['has_content_feature'] = (modeling_with_content['usable_metadata_watch_time'] > 0).astype(int)
modeling_with_content['no_content_feature_flag'] = (modeling_with_content['has_content_feature'] == 0).astype(int)

content_attach_check = pd.DataFrame([
    {'metric': 'modeling_usage_rows', 'value': int(len(modeling_usage))},
    {'metric': 'content_features_rows', 'value': int(len(content_features))},
    {'metric': 'modeling_with_content_rows', 'value': int(len(modeling_with_content))},
    {'metric': 'has_content_feature_rows', 'value': int(modeling_with_content['has_content_feature'].sum())},
    {'metric': 'no_content_feature_rows', 'value': int(modeling_with_content['no_content_feature_flag'].sum())},
])
content_attach_check.to_csv(TABLES_DIR / '05_content_feature_attach_check.csv', index=False, encoding='utf-8-sig')
display(content_attach_check)


## 5-15. 가설형 콘텐츠 세그먼트 후보 생성

이 단계에서는 최종 결론을 내리지 않는다. 06번 유의성 검정에서 테스트할 후보 flag만 만든다.


In [ ]:
def get_col(df, col, default=0):
    if col in df.columns:
        return df[col]
    return pd.Series(default, index=df.index)

kids_ratio = get_col(modeling_with_content, 'tag_ratio_애니메이션_키즈')
family_ratio = get_col(modeling_with_content, 'tag_ratio_가족')
all_age_ratio = get_col(modeling_with_content, 'rating_ratio_전체')
action_ratio = get_col(modeling_with_content, 'tag_ratio_액션')
sf_ratio = get_col(modeling_with_content, 'tag_ratio_SF_판타지')
thriller_ratio = get_col(modeling_with_content, 'tag_ratio_스릴러_범죄')
long_ratio = get_col(modeling_with_content, 'is_long_movie_ratio')
recent_ratio = get_col(modeling_with_content, 'is_recent_content_ratio')
w3_minus_w1 = get_col(modeling_with_content, 'w3_minus_w1_watch_time')
unique_days = get_col(modeling_with_content, 'unique_days')

modeling_with_content['family_content_affinity'] = (
    (kids_ratio >= 0.20) | (family_ratio >= 0.10) | (all_age_ratio >= 0.30)
).astype(int)

modeling_with_content['family_2screen_lifestyle'] = (
    (modeling_with_content['max_screen'] == 2)
    & (modeling_with_content['family_content_affinity'] == 1)
    & (unique_days >= 2)
).astype(int)

modeling_with_content['promo2_family_lifestyle_candidate'] = (
    (modeling_with_content['is_100won'] == 1)
    & (modeling_with_content['max_screen'] == 2)
    & (modeling_with_content['family_content_affinity'] == 1)
).astype(int)

modeling_with_content['action_sf_thriller_affinity'] = (
    ((action_ratio + sf_ratio + thriller_ratio) >= 0.40)
).astype(int)

modeling_with_content['premium_action_trial_risk'] = (
    (modeling_with_content['is_100won'] == 1)
    & (modeling_with_content['max_screen'] == 4)
    & (modeling_with_content['action_sf_thriller_affinity'] == 1)
    & ((long_ratio >= 0.30) | (w3_minus_w1 > 0))
).astype(int)

modeling_with_content['age40_2screen_anime_kids_candidate'] = (
    (modeling_with_content['is_100won'] == 1)
    & (modeling_with_content['max_screen'] == 2)
    & (modeling_with_content.get('is_user_verified', 0) == 1)
    & (modeling_with_content.get('age', 0).between(40, 49))
    & ((kids_ratio >= 0.30) | (modeling_with_content['top_genre_tag'].astype(str) == '애니메이션/키즈'))
).astype(int)

modeling_with_content['promo4_long_recent_trial_candidate'] = (
    (modeling_with_content['is_100won'] == 1)
    & (modeling_with_content['max_screen'] == 4)
    & (long_ratio >= 0.30)
    & (recent_ratio >= 0.20)
).astype(int)

candidate_flags = [
    'family_content_affinity',
    'family_2screen_lifestyle',
    'promo2_family_lifestyle_candidate',
    'action_sf_thriller_affinity',
    'premium_action_trial_risk',
    'age40_2screen_anime_kids_candidate',
    'promo4_long_recent_trial_candidate',
]

segment_rate_rows = []
for col in candidate_flags:
    tmp = modeling_with_content.groupby(col)['is_repurchase'].agg(n='count', repurchase_rate='mean').reset_index()
    tmp.insert(0, 'segment_flag', col)
    tmp = tmp.rename(columns={col: 'flag_value'})
    segment_rate_rows.append(tmp)
segment_rate_table = pd.concat(segment_rate_rows, ignore_index=True)
segment_rate_table.to_csv(TABLES_DIR / '05_content_feature_candidate_segment_rates.csv', index=False, encoding='utf-8-sig')
display(segment_rate_table)


## 5-16. 주요 집단별 콘텐츠 요약

이 표는 06번의 정식 유의성 검정 전에 보는 탐색적 요약이다.


In [ ]:
summary_cols = [
    'metadata_covered_watch_ratio', 'usable_metadata_watch_ratio',
    'tag_ratio_드라마', 'tag_ratio_액션', 'tag_ratio_스릴러_범죄', 'tag_ratio_SF_판타지',
    'tag_ratio_애니메이션_키즈', 'tag_ratio_가족',
    'rating_ratio_전체', 'rating_ratio_12세', 'rating_ratio_15세', 'rating_ratio_청불',
    'is_long_movie_ratio', 'is_recent_content_ratio', 'avg_runtime_weighted',
    'genre_tag_entropy_norm', 'country_entropy_norm',
]
summary_cols = [c for c in summary_cols if c in modeling_with_content.columns]

content_summary_by_100won = modeling_with_content.groupby('is_100won').agg(
    n=('membership_row_id', 'count'),
    repurchase_rate=('is_repurchase', 'mean'),
    **{f'avg_{c}': (c, 'mean') for c in summary_cols}
).reset_index()

content_summary_by_100won_screen = modeling_with_content.groupby(['is_100won', 'max_screen'], dropna=False).agg(
    n=('membership_row_id', 'count'),
    repurchase_rate=('is_repurchase', 'mean'),
    **{f'avg_{c}': (c, 'mean') for c in summary_cols}
).reset_index()

content_summary_by_100won.to_csv(TABLES_DIR / '05_content_feature_summary_by_100won.csv', index=False, encoding='utf-8-sig')
content_summary_by_100won_screen.to_csv(TABLES_DIR / '05_content_feature_summary_by_100won_maxscreen.csv', index=False, encoding='utf-8-sig')

display(content_summary_by_100won)
display(content_summary_by_100won_screen)


## 5-17. 산출물 저장


In [ ]:
PATH_USER_CONTENT_FEATURES = INTERIM_DIR / 'user_content_features.csv'
PATH_MODELING_WITH_CONTENT = PROCESSED_DIR / 'modeling_feature_table_with_content.csv'
PATH_CONTENT_SUMMARY_JSON = INTERIM_DIR / 'content_feature_summary.json'

content_features.to_csv(PATH_USER_CONTENT_FEATURES, index=False, encoding='utf-8-sig')
modeling_with_content.to_csv(PATH_MODELING_WITH_CONTENT, index=False, encoding='utf-8-sig')

summary = {
    'inputs': {k: str(v) for k, v in INPUT_FILES.items()},
    'outputs': {
        'user_content_features': str(PATH_USER_CONTENT_FEATURES),
        'modeling_feature_table_with_content': str(PATH_MODELING_WITH_CONTENT),
        'content_feature_summary': str(PATH_CONTENT_SUMMARY_JSON),
    },
    'rows': {
        'obs_view': int(len(obs_view)),
        'movie_metadata_v2': int(len(movie_meta)),
        'modeling_usage': int(len(modeling_usage)),
        'view_meta': int(len(view_meta)),
        'content_features': int(len(content_features)),
        'modeling_with_content': int(len(modeling_with_content)),
    },
    'coverage': {
        'watch_time_total': float(view_meta['watch_time'].sum()),
        'watch_time_metadata_covered': float(view_meta.loc[view_meta['metadata_covered_flag'] == 1, 'watch_time'].sum()),
        'watch_time_usable_for_content_features': float(view_meta.loc[view_meta['use_for_content_features'] == 1, 'watch_time'].sum()),
    },
    'feature_counts': {
        'content_feature_columns': int(len(content_features.columns)),
        'modeling_with_content_columns': int(len(modeling_with_content.columns)),
    },
    'candidate_flags': candidate_flags,
}

with open(PATH_CONTENT_SUMMARY_JSON, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('saved:', PATH_USER_CONTENT_FEATURES)
print('saved:', PATH_MODELING_WITH_CONTENT)
print('saved:', PATH_CONTENT_SUMMARY_JSON)


## 5-18. 최종 검산


In [ ]:
assert len(modeling_with_content) == len(modeling_usage), '05번 결합 후 행 수는 04번 모델링 테이블과 같아야 한다.'
assert modeling_with_content['membership_row_id'].nunique() == modeling_usage['membership_row_id'].nunique(), 'membership_row_id 고유 수가 보존되어야 한다.'
assert PATH_USER_CONTENT_FEATURES.exists(), 'user_content_features.csv 저장 실패'
assert PATH_MODELING_WITH_CONTENT.exists(), 'modeling_feature_table_with_content.csv 저장 실패'

final_check = pd.DataFrame([
    {'check': 'row_count_preserved_from_04', 'value': len(modeling_with_content) == len(modeling_usage)},
    {'check': 'membership_row_id_unique_preserved', 'value': modeling_with_content['membership_row_id'].nunique() == modeling_usage['membership_row_id'].nunique()},
    {'check': 'content_feature_file_exists', 'value': PATH_USER_CONTENT_FEATURES.exists()},
    {'check': 'modeling_with_content_file_exists', 'value': PATH_MODELING_WITH_CONTENT.exists()},
    {'check': 'has_content_feature_rows_gt_zero', 'value': int(modeling_with_content['has_content_feature'].sum()) > 0},
])
final_check.to_csv(TABLES_DIR / '05_content_feature_final_checks.csv', index=False, encoding='utf-8-sig')
display(final_check)


## 5-19. 05번 노트북 결론

05번 노트북의 최종 산출물은 `modeling_feature_table_with_content.csv`이다. 이 파일은 04번 사용 행동 피처 테이블에 영화 메타데이터 기반 콘텐츠 성향 피처를 붙인 모델링 후보 테이블이다.

다음 단계는 `06_significance_tests.ipynb`이다. 06번에서는 04번 행동 피처와 05번 콘텐츠 피처를 모두 포함해, 전체 고객, 100원딜 고객, 100원딜+2인, 100원딜+4인 등 주요 분석군별로 p-value, FDR 보정 p-value, 효과크기, 재구독률 차이를 계산한다.
